# Module 5 — Feature Engineering & Data Preprocessing
Reuses `preprocessing.py`, `eda_utils.py`, and `src/feature_engineering.py`.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd() / "src"))
from feature_engineering import (
    build_features, save_features, validate_dataset, FEATURES_PATH,
)
from eda_utils import load_cleaned_data


## Task 1-5: Build Features, Encode, Select, Transform, Scale

In [2]:
df_before = load_cleaned_data()
print("Before:", df_before.shape)

df_final, skew_summary = build_features()
print("After :", df_final.shape)
df_final.head()


Before: (2237, 30)
After : (2237, 52)


,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,...,Marital_Status_Widow,Preferred_Shopping_Channel_Catalog,Preferred_Shopping_Channel_Store,Preferred_Shopping_Channel_Web,Product_Preference_Fish,Product_Preference_Fruits,Product_Preference_Gold,Product_Preference_Meat,Product_Preference_Sweets,Product_Preference_Wine
0,0.300178,-0.825388,-0.930227,0.307272,0.987474,1.433573,1.395519,1.580190,1.411011,1.062551,...,0,1,0,0,0,0,0,0,0,1
1,-0.263278,1.032151,0.906417,-0.383584,-1.214255,-0.984954,-1.398673,-0.866215,-0.970709,-0.912889,...,0,0,1,0,0,0,0,0,0,1
2,0.943942,-0.825388,-0.930227,-0.798098,0.766533,1.066149,0.464396,1.317856,0.534002,0.497407,...,0,0,1,0,0,0,0,0,0,1
3,-1.204345,1.032151,-0.930227,-0.798098,-1.214255,-0.401083,-0.694369,-0.082288,-0.535750,-1.032648,...,0,0,1,0,0,0,0,1,0,0
4,0.307583,1.032151,-0.930227,1.550812,0.268696,0.984692,0.417656,0.793933,0.685334,-0.270644,...,0,0,1,0,0,0,0,0,0,1


## Task 2: Encoding Summary
Categorical columns one-hot encoded; `Customer_Activity_Level` label-encoded (ordinal).

In [3]:
encoded_cols = [c for c in df_final.columns if c.startswith(
    ("Education_", "Marital_Status_", "Preferred_Shopping_Channel_", "Product_Preference_")
)]
print(f"Original categorical columns: Education, Marital_Status, Preferred_Shopping_Channel, Product_Preference")
print(f"Resulting one-hot columns ({len(encoded_cols)}): {encoded_cols}")
print("Customer_Activity_Level encoded as ordinal: Inactive=0, Moderate=1, Active=2")


Original categorical columns: Education, Marital_Status, Preferred_Shopping_Channel, Product_Preference
Resulting one-hot columns (19): ['Education_2N Cycle', 'Education_Basic', 'Education_Graduation', 'Education_Master', 'Education_Phd', 'Marital_Status_Divorced', 'Marital_Status_Married', 'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Widow', 'Preferred_Shopping_Channel_Catalog', 'Preferred_Shopping_Channel_Store', 'Preferred_Shopping_Channel_Web', 'Product_Preference_Fish', 'Product_Preference_Fruits', 'Product_Preference_Gold', 'Product_Preference_Meat', 'Product_Preference_Sweets', 'Product_Preference_Wine']
Customer_Activity_Level encoded as ordinal: Inactive=0, Moderate=1, Active=2


## Task 4: Skewness & Transformation Summary
Log1p applied where |skew| > 0.75.

In [4]:
skew_summary

,Feature,Skew_Before,Skew_After,Transformed
0,Income,0.090,0.090,False
1,MntWines,1.176,-0.549,True
2,MntFruits,2.104,0.083,True
3,MntMeatProducts,2.084,-0.084,True
4,MntFishProducts,1.919,-0.052,True
5,MntSweetProducts,2.135,0.085,True
6,MntGoldProds,1.885,-0.343,True
7,Total_Spending,0.860,-0.372,True
8,Avg_Spending_Per_Purchase,22.200,-0.087,True


## Task 5: Feature Scaling
StandardScaler applied to all non-binary numeric columns (mean 0, unit variance) — the standard default for distance-based clustering algorithms such as K-Means.

In [5]:
scaled_preview = df_final.select_dtypes(include="number").describe().loc[["mean", "std"]].T
scaled_preview.head(10)


,mean,std
Income,-3.176320e-17,1.000224
Kidhome,1.429344e-17,1.000224
Teenhome,-9.528959e-17,1.000224
Recency,1.195090e-16,1.000224
MntWines,4.224505e-16,1.000224
MntFruits,5.479152e-17,1.000224
MntMeatProducts,-9.052511e-17,1.000224
MntFishProducts,-1.699331e-16,1.000224
MntSweetProducts,3.176320e-17,1.000224
MntGoldProds,1.977259e-16,1.000224


## Task 8: Final Validation

In [6]:
validate_dataset(df_final)

,Check,Result
0,No missing values,True
1,Duplicate rows (post ID removal),184
2,All columns numeric,True
3,No identifier columns present,True


## Save Engineered Dataset

In [7]:
path = save_features(df_final)
print(f"Saved: {path}")


Saved: /home/xploit/Downloads/ML-Projects/Day-6/03_Cleaned_Data/customer_personality_features.csv
